RNN 循环神经网络：从序列建模到 LSTM 和 GRU

循环神经网络（Recurrent Neural Network，RNN）用于处理有顺序关系的数据，例如文本、语音、时间序列、用户行为和传感器读数。与把每条样本视为独立向量的 MLP 不同，RNN 会在每个时间步维护隐藏状态（hidden state），让当前输出同时依赖当前输入和此前序列的信息。

本文从序列张量和手写循环开始，逐步学习 `nn.RNN`、`nn.LSTM`、`nn.GRU`、可变长度 batch、梯度消失、截断反向传播和完整训练流程。示例默认使用 PyTorch 2.x：


In [1]:
import torch
from torch import nn
import torch.nn.functional as F

In [4]:
# 一个长度为 4、每步含 2 个特征的单样本时间序列可写作：
x = torch.tensor([
    [0.1, 1.0],  # t=0
    [0.2, 0.8],  # t=1
    [0.3, 0.7],  # t=2
    [0.4, 0.5],  # t=3
])
print(x.shape)  # torch.Size([4, 2])，即 [T, D]
# 这里 `T` 是序列长度（时间步数），`D` 是每个时间步的输入特征数。`T` 不是 batch size；不同样本的 `T` 可以不同。




torch.Size([4, 2])
torch.Size([8, 10, 5])


In [11]:
## 3. RNN 的输入输出形状
"""
PyTorch 的循环层默认采用 sequence-first 格式：

输入 input:  [T, N, D]
输出 output: [T, N, H * directions]
最终状态 h_n: [layers * directions, N, H]

T: sequence length，时间步数
N: batch size
D: input_size，每个时间步输入特征数
H: hidden_size，隐藏状态特征数
"""


# 若设置 `batch_first=True`，输入和输出改为 `[N, T, D]` 与 `[N, T, H * directions]`；**最终隐藏状态 `h_n` 的 shape 不变**。初学时建议明确选择一种格式，以下多数 batch 示例使用 `batch_first=True`：
# `batch_first=True` 不改变“时间沿 T 维依次处理”的本质，只改变张量维度顺序。对于未分 batch 的单条序列，通常仍建议先 `unsqueeze(0)` 添加 batch 维。

# 以下是设置 batch_first=True时的数据形状
batch = torch.randn(3, 5, 2)  # [N=3, T=5, D=2]
print(batch.shape)

# 有8个批次数据，每个批次有10个时间点，每个样本数据有16个特征
x2 =  torch.randn([8, 10, 16]) # [N=8, T=10, D=16]
print(x2.shape)

torch.Size([3, 5, 2])
torch.Size([8, 10, 16])


In [12]:
## 4. 时间步与隐藏状态

# 一个长度为3每个时间点一个特征的序列
sequence = torch.tensor([[1.0], [2.0], [3.0]])  # [T, D=1]
# 初始隐藏层为一个全0的序列
hidden = torch.zeros(1)                         # h_0

# 将这个序列进行传播
for t, x_t in enumerate(sequence):
    # x_t + hidden 表示当前的序列数据再加上上一时刻的隐藏层输出，结果表示当前时间的序列输入（因为RNN就是当前时刻的原始输入➕上一时刻的隐藏层输出作为当前时刻的模型输入）
    # torch.tanh 是激活函数，结果表示道歉RNN模型输出后的隐藏层结果
    # 注意，每次循环必须依赖上一次的hidden，否则rnn将蜕变为只能处理当前时刻的数据，无法依赖前面时间序列的数据
    hidden = torch.tanh(x_t + hidden)
    # t是索引，此时可以表示为时间点，x_t是当前时刻的输入，h_t是当前时刻隐藏的输出
    print(f"t={t}, x_t={x_t.item():.1f}, h_t={hidden.item():.4f}")


t=0, x_t=1.0, h_t=0.7616
t=1, x_t=2.0, h_t=0.9920
t=2, x_t=3.0, h_t=0.9993


In [15]:
## 5. 输出来自哪里：全部时间步还是最后时间步
# RNN 通常可获得每个时间步的输出，也可只使用最后有效时间步的信息：

outputs = torch.randn(4, 6, 8)  # [N=4, T=6, H=8]

all_steps = outputs             # 用于序列标注，shape [4, 6, 8]
last_step = outputs[:, -1, :]   # 用于固定长度序列分类，shape [4, 8]
print(all_steps.shape, last_step.shape)
# 对等长、未填充的序列，`outputs[:, -1, :]` 可以表示最后时间步
# 对含 padding 的变长序列，最后一个位置可能是 padding，不能直接这样取；第 16 节会给出按真实长度取得最后有效输出的方法。


torch.Size([4, 6, 8]) torch.Size([4, 8])


In [31]:
# 实现一个简单的RNN Cell
# RNN 将当前时刻的输入信息和上一时刻的历史信息分别进行线性变换，然后相加，最后通过 tanh 激活函数，得到当前时刻的隐藏状态。
# RNN的简单理解可以为：当前隐藏状态 = 当前输入的信息 + 历史记忆信息
class SimpleRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(SimpleRNNCell, self).__init__()
        # RNN 在每个时间步需要处理两种不同来源的信息，所以要定义两个不同的Linear
        # input_to_hidden 负责将当前输入 x_t 转换为隐藏层需要的特征。
        # hidden_to_hidden 负责将上一时刻的隐藏状态转换为当前时刻需要的历史信息。

        # in_features 根据实际数据的输入特征数定义，out_features 这个必须使用hidden_size，因此输入特征数据经过input_to_hidden线性层之后这个输出必须要和hidden_to_hidden的输出一致，这样才能进行相同位置相加，之后经过激活函数的到最终输出
        self.input_to_hidden = nn.Linear(in_features = input_size, out_features = hidden_size)
        # 这里的 in_features 和 out_features 必须一致，因为这个hidden_to_hidden是一个隐藏层到隐藏层的线性层，这一时刻的输出要送到下一时刻去使用，因此形状必须一致
        self.hidden_to_hidden = nn.Linear(in_features=hidden_size, out_features=hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        """
        :param x_t:     [N, D] 是当前时刻的输入
        :param h_prev:  [N, H] 是上一时刻的隐藏层输出，这个就是为什么RNN能记住原来信息的原因
        :return:        [N, H] 返回这一时刻的输入和上一时刻隐藏层输出经过激活函数后的结果
        """
        # x_t 是 [N, D]。 h_prev 是 [N, H]
        # print(f"self.input_to_hidden(x_t).shape={self.input_to_hidden(x_t).shape}")         # [3, 4]
        # print(f"self.hidden_to_hidden(h_prev).shape={self.hidden_to_hidden(h_prev).shape}") # [3, 4]
        # 输入输出分别进过不同线性层（并行），结果相同位置相加之后再经过激活函数，最后得到最后输出。即两种信息需要独立变换后融合
        # 并行：是因为当前隐藏层要同时考虑当前这一层的输入x_t 和 上一层隐藏层的输出h_prev，之后两种信息应该在同一个隐藏空间中融合。
        # torch.tanh；引入非线性，将激活层输出限制在[-1,1]之间
        return torch.tanh(self.input_to_hidden(x_t) + self.hidden_to_hidden(h_prev))


cell = SimpleRNNCell(input_size=2, hidden_size=4)
x_t = torch.randn(3, 2)     # [3,2] ，3表示批量，2表示输入一个样本是2个特征，这个2与定义模型时的input_size=2对应
h_prev = torch.zeros(3, 4)  # [3,4] ，3表示批量，4表示隐藏层有4个特征，这个4与定义模型时的hidden_size=4对应
h_t = cell(x_t, h_prev)
print(h_t.shape)  # [3, 4]


print("遍历多个时间步的数据")
batch_size = 3
seq_len = 6
num_features = 2
hidden_size = 4
h_prev = torch.zeros(batch_size, 4)
rnn_cell = SimpleRNNCell(input_size=num_features, hidden_size=hidden_size)
x_seq = torch.randn(batch_size, seq_len, num_features)
step_outputs = []
# 遍历每一个时间步输入rnn_cell
for t in range(seq_len):
    # 第一个维度是批量，第二个维度是时间序列，第三个维度是特征
    # 所以第一、三维度数据取全部，第二维度的话就取当前时间点的数据
    x_t = x_seq[:, t, :]
    # h_prev 不断更新 h_prev，使之能够学习每一个时间步的数据
    # 每次循环都会将上一时刻的隐藏状态传给下一时刻。
    # 这就是 RNN 的核心：当前输入不断变化，隐藏状态不断传递，但模型参数始终共享。
    # RNN 在不同时间步执行的是同一种计算规则，输入数据虽然不同，但数据的特征含义和隐藏状态的维度保持一致，因此可以使用同一套权重参数。
    h_prev = rnn_cell(x_t, h_prev)
    step_outputs.append(h_prev)
    print(f"第 {t + 1} 个时间步，h.shape: {h_prev.shape}")
    # rnn时共享权重id的
    print(
        f"第 {t + 1} 个时间步，"
        f"weight id: {id(rnn_cell.input_to_hidden.weight)}, "
        f"hidden weight id: {id(rnn_cell.hidden_to_hidden.weight)}"
    )
outputs = torch.stack(step_outputs, dim=1) # [3, 6, 4] , 注意里的[3,4]是一个时间步的输出，因为这里把所有时间步的数据都堆叠了，所以这里的输出第1维的6是指6个时间步
print("hidden.shape: ", h_prev.shape)
print("batch stp shape:", outputs.shape)


torch.Size([3, 4])
遍历多个时间步的数据
第 1 个时间步，h.shape: torch.Size([3, 4])
第 1 个时间步，weight id: 4477227472, hidden weight id: 4477227392
第 2 个时间步，h.shape: torch.Size([3, 4])
第 2 个时间步，weight id: 4477227472, hidden weight id: 4477227392
第 3 个时间步，h.shape: torch.Size([3, 4])
第 3 个时间步，weight id: 4477227472, hidden weight id: 4477227392
第 4 个时间步，h.shape: torch.Size([3, 4])
第 4 个时间步，weight id: 4477227472, hidden weight id: 4477227392
第 5 个时间步，h.shape: torch.Size([3, 4])
第 5 个时间步，weight id: 4477227472, hidden weight id: 4477227392
第 6 个时间步，h.shape: torch.Size([3, 4])
第 6 个时间步，weight id: 4477227472, hidden weight id: 4477227392
hidden.shape:  torch.Size([3, 4])
batch stp shape: torch.Size([3, 6, 4])


In [36]:
# SimpleRNNCell 这只是一个 rnn 单元，要想变成真正的RNN还要进行堆叠
# 定义一个 自定义简单的 RNN 网络
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn_cell = SimpleRNNCell(input_size, hidden_size)

    def forward(self, x, h_0=None):
        """
            rnn 前向传播
        :param x_t: [N, T, D] 一个批次的序列
        :param h_0: [N, H] 初始化的隐藏层之台
        :return:
            output: [N, T, H]
            h_n: [num_layers, N, H]
        """

        # 获取输入数据形状
        batch_size, seq_len, num_features = x.shape

        # 初始化隐藏层之台
        if h_0 is None:
            h = torch.zeros(batch_size, self.hidden_size).to(x.device)
        else:
            h = h_0

        outputs = []
        # 遍历每一个时间步
        for t in range(seq_len):
            x_t = x[:, t, :]
            # 顺序输入每一个 rnn_cell
            h = self.rnn_cell(x_t, h)
            outputs.append(h)

        # 堆叠输出
        output = torch.stack(outputs, dim=1)

        # 获取最后一个隐藏层状态，表示整个序列的最终状态
        h_last = output[:, -1, :]

        return output, h_last

# 测试 SimpleRNN
num_features = 2
hidden_size = 4
num_layers = 2
batch_size = 3
seq_len = 6
x = torch.randn(batch_size, seq_len, num_features)       # [N, T, D]
rnn = SimpleRNN(num_features, hidden_size, num_layers=num_layers)
output, h_n = rnn(x)
print(output.shape)  # [3, 6, 4]，每个时间步最后层输出
print(h_n.shape)     # [3, 4]，最后时间步的隐藏状态




torch.Size([3, 6, 4])
torch.Size([3, 4])


In [37]:
## 7. `nn.RNN`：基础循环层
rnn = nn.RNN(
    input_size=2,
    hidden_size=4,
    num_layers=1, # 有几个rnn层进行堆叠
    nonlinearity="tanh", # 使用的激活函数，可选值：'tanh' or 'relu'
    batch_first=True, # 指定输入输出的形状为：[批量N, 时间序列T, 特征D]，
    bidirectional=False, # 是否建立正向和反向两个方向
)

x = torch.randn(3, 5, 2)       # [N, T, D]
output, h_n = rnn(x)
print(output.shape)  # [3, 5, 4]，每个时间步最后层输出
print(h_n.shape)     # [1, 3, 4]，最后时间步的隐藏状态


torch.Size([3, 5, 4])
torch.Size([1, 3, 4])


In [40]:
## 8. 基于 `nn.RNN` 的序列分类
# 下例以每条序列的最后隐藏状态进行三分类。模型输出原始 logits，应直接交给 `CrossEntropyLoss`：
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(RNNClassifier, self).__init__()
        # rnn net
        self.rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True, nonlinearity="tanh")
        # linear 分类器
        self.linear = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h_n = self.rnn(x) # h_n.shape=[1, N, H],
        # `h_n[-1]` 代表最后一层、正向 RNN 的最终隐藏状态。它比 `output[:, -1]` 更直接表达“最后一步状态”，但对变长且未 packed 的输入，两者同样会受 padding 影响。
        last_layer_hidden = h_n[-1]       # last_layer_hidden.shape=[N, H]
        return self.linear(last_layer_hidden) # 输出 [N, num_classes]

model = RNNClassifier(input_size=6, hidden_size=32, num_classes=3)
x = torch.randn(8, 12, 6) # 批次8，时间序列12，特征数6
labels = torch.randint(0, 3, (8,)) # 输出 [8]，每一个值都是0/1/2中的一个表示类别
logits = model(x)
loss = nn.CrossEntropyLoss()(logits, labels)
print(logits.shape, loss.item())  # [8, 3]，标量 loss


torch.Size([8, 3]) 1.1768205165863037


In [42]:
## 9. 多层 RNN 与 dropout
# 多层 RNN 把前一层每个时间步的输出序列作为下一层的输入序列：

rnn = nn.RNN(
    input_size=10,
    hidden_size=16,
    num_layers=3,
    dropout=0.2,
    batch_first=True,
)
# `dropout=0.2` 放在层与层之间，不会作用于最后一层输出；当 `num_layers=1` 时该参数不生效。不要假设它会在每个时间步、每个状态连接上做 dropout。需要更细粒度的正则化时，可在循环层前后显式加 `nn.Dropout`，但要先确认是否符合任务语义。

x = torch.randn(4, 7, 10) # 4个批量，7个时间序列，10个特征数据
output, h_n = rnn(x)
print(output.shape)  # [4, 7, 16]
print(h_n.shape)     # [3, 4, 16]，每层的最终状态, 3表示rnn层数，4表示批量，16表示隐藏层大小

torch.Size([4, 7, 16])
torch.Size([3, 4, 16])


In [59]:
## 10. 通过时间的反向传播 BPTT
# RNN 的参数在时间上共享，损失对某个参数的梯度来自所有时间步。反向传播会从后续时间步沿隐藏状态依赖逐步传回早期步骤，这称为随时间反向传播 Backpropagation Through Time（BPTT）。
# 普通全连接神经网络是不同网络层通常使用不同参数；RNN 是同一个循环单元在不同时间步重复执行，并共享同一套参数。它通过隐藏状态传递历史信息，通过权重共享学习通用的序列处理规则。
# 理解了这一点，下一步就可以学习 BPTT：RNN 是如何沿着时间轴反向传播梯度的，以及为什么普通 RNN 会出现梯度消失和梯度爆炸。
# PyTorch 会在前向传播时建立计算图，`loss.backward()` 自动完成 BPTT：

rnn = nn.RNN(1, 8, batch_first=True) # rnn
head = nn.Linear(8, 1)  # 分类器
x = torch.randn(2, 6, 1) # 特征
target = torch.randn(2, 1) # 标签（回归值）

_, h_n = rnn(x) # 前向传播
prediction = head(h_n[-1]) # prediction.shape=(2,1) 预测值
loss = F.mse_loss(prediction, target) # 计算损失
# loss = nn.MSELoss()(prediction, target)
loss.backward() # 反向传播
print(rnn.weight_hh_l0.grad.norm().item())  # 查看梯度

1.2171581983566284


In [61]:
## 11. 梯度消失与梯度爆炸
# 在 BPTT 中，梯度需要反复乘以循环权重和激活函数的局部导数。
# 大量绝对值小于 1 的因子相乘会让早期时间步的梯度趋近于零，即梯度消失；
# 大量大于 1 的因子相乘则可能让梯度急剧增大，造成梯度爆炸。

# 梯度裁剪应在 `backward()` 后、`optimizer.step()` 前执行：
# loss.backward()
# grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
# optimizer.step()
# print(float(grad_norm))  # 裁剪前的总梯度范数

In [65]:
## 12. 截断反向传播 TBPTT

# 对极长流式序列，若一直把隐藏状态连接到过去，计算图会无限增长。截断反向传播（Truncated BPTT，TBPTT）会在固定长度片段后保留数值状态、切断其到旧计算图的梯度连接：

rnn = nn.GRU(input_size=8, hidden_size=32, batch_first=True)
hidden = None
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
stream_chunks = torch.randn(16, 2, 8)
chunk_labels = torch.randn(16, 1)
for (chunk, labels) in zip(stream_chunks, chunk_labels):  # 每段 shape: [N, chunk_length, 8]
    output, hidden = rnn(chunk, hidden)
    loss = criterion(output, labels)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rnn.parameters(), 1.0) # 梯度裁剪防止梯度爆炸
    optimizer.step()

    # `detach()` 不会把隐藏状态清零，模型仍能在前向中使用前一片段的信息；它只限制梯度不再跨越片段边界。TBPTT 是内存与长期梯度传播能力之间的权衡，片段长度应由任务依赖范围和资源决定。
    hidden = hidden.detach()  # 保留数值，但切断跨 chunk 的反向图


In [69]:
## 13. `nn.LSTM`：门控记忆
# 长短期记忆网络（Long Short-Term Memory，LSTM）在隐藏状态 `h_t` 之外维护细胞状态 `c_t`。它用输入门、遗忘门、输出门控制什么信息写入、保留和读取，从而缓解普通 RNN 的长期梯度问题。
# 门值在 0 到 1 之间，逐元素决定信息流。不要把门理解为硬性的 if 语句；它们是从数据中学习到的连续权重。
# LSTM（Long Short-Term Memory，长短期记忆网络）是 RNN 的一种改进和扩展，也可以理解为 RNN 的一种特殊结构。
# LSTM 完全可以独立使用，不需要先创建一个普通 RNN。

# input_size 输入特征数量
# hidden_size 隐藏层维度
# num_layers 有多少个lstm块堆叠
lstm = nn.LSTM(input_size=5, hidden_size=12, num_layers=2, batch_first=True)
x = torch.randn(4, 7, 5) # 4个批量，7个时间序列，5个特征数据
# output 表示数据经过lstm之后的输出
# hidden_n 表示最终隐藏层输出
# cell_n 表示最终的细胞单元输出，专门负责长期信息传递，门控细胞，控制哪些信息要记忆，哪些信息不要记忆的
output, (hidden_n, cell_n) = lstm(x)

print(output.shape)  # [4, 7, 12] ，4个批次，7个时间序列，12个隐藏层输出
print(hidden_n.shape)     # [2, 4, 12]，每层最终隐藏状态， 2表示lstm层数num_layers=2，4表示批次，12表示隐藏层输出
print(cell_n.shape)     # [2, 4, 12]，每层最终细胞状态



torch.Size([4, 7, 12])
torch.Size([2, 4, 12])
torch.Size([2, 4, 12])


In [82]:
## 14. 基于 LSTM 的情感分类结构
# 文本 token ID 不能直接作为连续数字送入 RNN，应先经 `nn.Embedding` 转为可学习向量。下例输入已 padding 到相同长度，暂不处理变长问题：
class LSTMTextClassifier(nn.Module):
    def __init__(self, vocab_size, embdding_dim, hidden_size, num_classes):
        super(LSTMTextClassifier, self).__init__()
        # 嵌入层 -> LSTM -> 线性分类层
        # 在深度学习中，一个完整模型经常被划分为两部分：
        # - Backbone（骨干网络）：负责提取特征、学习数据表示。 【这里的 embedding 和 lstm】
        # - Head（输出头）：负责把提取到的特征转换成具体任务的预测结果。 【这里的 head】 head 表示	模型的任务输出部分。 head 是泛指，而具体的classifier、regressor等是一种具体的head
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embdding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embdding_dim, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(in_features=hidden_size, out_features=num_classes) # 模型的任务输出头（Prediction Head / Task Head）

    def forward(self, token_ids):
        # token_ids 表示文本在词汇表里面的id ,
        # 获取词汇id对应的嵌入数据
        embeddings = self.embedding(token_ids) # 输入：[N, T]， 输出：[N, T, E]，其中，N表示批次大小，T表示时间序列对应的token_idE表示嵌入维度大小
        # 将嵌入向量输入lstm
        outputs, (h_n, c_n) = self.lstm(embeddings) # 分别表示输出，最终隐藏层状态，最终门控细胞状态
        print(f"outputs:{outputs.shape}, h_n: {h_n.shape}, c_n: {c_n.shape}")
        return self.head(h_n[-1]) # 使用 h_n 得到最终的类别输出，h_n 是所有时刻的记忆信息，包含所有输入信息，用这个来做分类

lstmClassifier = LSTMTextClassifier(vocab_size=100, embdding_dim=64, hidden_size=32, num_classes=2)
token_ids = torch.randint(0, 100, size=(4, 10)) # size=(4, 10) 表示4个批次，每个批次有10个token_id序列
labels = torch.tensor([0, 1, 1, 0])
logits = lstmClassifier(token_ids) # logits 模型原始输出的分数
print(f"logits: {logits}") # [4, 2] 表示 4个批次，每个批次里面有2列概率分数，那一列更接近1，那么预测的就是那一列的概率比较高
proba = F.log_softmax(logits, dim=1)
print(f"proba:{proba}")
pred_tagets = logits.argmax(1)
print(f"pred_tagets: {pred_tagets}")
loss = nn.CrossEntropyLoss()(logits, labels) # CrossEntropyLoss 内部已经包含了：F.log_softmax(pred, dim=1)
print(loss.item())



outputs:torch.Size([4, 10, 32]), h_n: torch.Size([1, 4, 32]), c_n: torch.Size([1, 4, 32])
logits: tensor([[-0.0481,  0.2484],
        [ 0.0744,  0.1908],
        [-0.1359, -0.0165],
        [-0.0447,  0.1403]], grad_fn=<AddmmBackward0>)
proba:tensor([[-0.8523, -0.5558],
        [-0.7530, -0.6367],
        [-0.7546, -0.6352],
        [-0.7899, -0.6049]], grad_fn=<LogSoftmaxBackward0>)
pred_tagets: tensor([1, 1, 1, 1])
0.7285297513008118


In [86]:
## 15. `nn.GRU`：更紧凑的门控循环层
# 门控循环单元（Gated Recurrent Unit，GRU）使用更新门和重置门控制状态，通常没有单独的细胞状态，因此参数比同尺寸 LSTM 少

gru = nn.GRU(input_size=5, hidden_size=12, num_layers=2, batch_first=True)
x = torch.randn(4, 7, 5)
output, h_n = gru(x)
print(output.shape)  # [4, 7, 12]
print(h_n.shape)     # [2, 4, 12]


# RNN、LSTM、GRU 的主要差别：
#
# | 层 | 状态 | 优点 | 局限 |
# | --- | --- | --- | --- |
# | `nn.RNN` | `h_t` | 简单、参数少、适合教学和短依赖 | 长依赖更易梯度消失/爆炸 |
# | `nn.LSTM` | `h_t` 与 `c_t` | 门控完善，常用于较长依赖 | 参数更多、状态接口更复杂 |
# | `nn.GRU` | `h_t` | 比 LSTM 紧凑，常是强基线 | 与 LSTM 没有普适的绝对优劣 |


# 注意：h_n 或 c_n 都是给RNN\LSTM\GRU内部使用的记忆数据，最终输出的时候外部是拿不到这些数据的


torch.Size([4, 7, 12])
torch.Size([2, 4, 12])


In [91]:
## 16. 可变长度序列与 padding
# 一个 batch 需要组成规则 Tensor，因此不同长度序列常被补到相同长度。默认补零并不等于模型会自动忽略零；如果零是合法输入，或 RNN 继续处理 padding，会污染末尾状态和损失。

# 先用 `pad_sequence` 构造 batch：

from torch.nn.utils.rnn import pad_sequence

sequences = [
    torch.tensor([4, 7, 2]),
    torch.tensor([5, 1]),
    torch.tensor([9, 3, 8, 6]),
]
# sequences 的问题是：每条序列长度不一样，无法直接使用 torch.tensor() 将它们转换成一个二维 Tensor，这时就需要 pad_sequence 来填充短序列。
# pad_sequence 流程：找到所有序列中最长的序列，将其他小于最长序列的序列长度填充到与最长序列同长度，不足的位置使用0来填充
padded = pad_sequence(sequences, batch_first=True, padding_value=0)
print(padded.shape)  # [3, 4]
print(padded)

# pad_sequence的适用场景是，后续同一个批次里面的 token_id 序列长度不一样的时候，就需要将每个批次的序列长度对齐，这样才能输入到模型中，所以就是要使用 pad_sequence 来将序列填充到同一长度


torch.Size([3, 4])
tensor([[4, 7, 2, 0],
        [5, 1, 0, 0],
        [9, 3, 8, 6]])


In [ ]:

# pad_sequence 和 pack_padded_sequence 的区别它们经常配合使用。
# 函数	作用
# pad_sequence	将不同长度的序列补齐到统一长度
# pack_padded_sequence	将补齐后的序列打包，让 RNN/LSTM 忽略 Padding 部分的无效计算
# pad_packed_sequence	将打包后的序列还原为补齐形式

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)

sequences = [
    torch.tensor([4, 7, 2]),
    torch.tensor([5, 1]),
    torch.tensor([9, 3, 8, 6]),
]

# 1. 补齐
padded = pad_sequence(
    sequences,
    batch_first=True,
    padding_value=0
)

# 2. 记录每条序列的真实长度
lengths = torch.tensor([3, 2, 4])

# 3. 打包
packed = pack_padded_sequence(
    padded,
    lengths,
    batch_first=True,
    enforce_sorted=False
).long()

# 4. 输入 LSTM
lstm = nn.LSTM(
    input_size=1,
    hidden_size=4,
    batch_first=True
)

# 这里仅演示 API 形式，实际文本任务应先经过 Embedding
x = packed
output, (h_n, c_n) = lstm(x)

# 5. 还原
output, output_lengths = pad_packed_sequence(
    output,
    batch_first=True,
    padding_value=0
)